# Emotion-Conditioned TTS: Fine-tuning LongCat-AudioDiT-1B on MELD

**Author:** ML Engineer — Speech Synthesis & Generative AI  
**Environment:** Kaggle T4 GPU (16 GB VRAM)  
**Model:** `meituan-longcat/LongCat-AudioDiT-1B`  
**Dataset:** MELD (Multimodal EmotionLines Dataset)

## Pipeline Overview
```
MELD (.mp4 + .csv)
  → Audio extraction & resampling (24 kHz mono)
  → Emotion-prefixed text prompts  ➜  [JOY] Hello!
  → MelSpectrogram feature extraction
  → LoRA-augmented AudioDiT (fp16 + gradient checkpointing)
  → Diffusion loss training
  → MCD evaluation & Mel-spectrogram comparison
```

## 0 · Requirements

In [ ]:
import subprocess, sys, os

def pip(*args, **kw):
    subprocess.run([sys.executable, '-m', 'pip'] + list(args), check=True, **kw)
print('Removing torchao to avoid torch-version conflict …')
pip('uninstall', '-y', 'torchao')
PACKAGES = ['librosa>=0.10.1', 'diffusers>=0.27.0', 'transformers>=4.40.0', 'peft>=0.10.0', 'accelerate>=0.29.0', 'bitsandbytes>=0.43.0', 'pymcd>=0.1.3', 'soundfile>=0.12.1', 'einops>=0.7.0']
pip('install', '-q', '--upgrade', *PACKAGES)
REPO_DIR = '/kaggle/working/LongCat-AudioDiT'
if not os.path.exists(REPO_DIR):
    print('Cloning LongCat-AudioDiT repo …')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/meituan-longcat/LongCat-AudioDiT.git', REPO_DIR], check=True)
    print('✅ Repo cloned.')
else:
    print('✅ Repo already present, skipping clone.')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Upgrading transformers to >=5.3.0 (required by audiodit) …')
pip('install', '-q', '--upgrade', 'transformers>=5.3.0')
print('✅ audiodit ready.')
import importlib.util
if importlib.util.find_spec('torchao') is not None:
    pip('uninstall', '-y', 'torchao')
    print('⚠  torchao was re-installed by a dependency; removed again.')
else:
    print('✅ torchao absent — PEFT will skip torchao dispatcher cleanly.')
print('✅ All packages ready.')


## 1 · Imports & Global Configuration

In [ ]:
import os
import json
import warnings
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from scipy.io import wavfile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, T5Tokenizer, T5EncoderModel, get_cosine_schedule_with_warmup
from diffusers import DDPMScheduler, DDIMScheduler, AutoencoderKL
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
try:
    from pymcd.mcd import Calculate_MCD
    MCD_AVAILABLE = True
except ImportError:
    MCD_AVAILABLE = False
    print('⚠  pymcd not found – MCD evaluation will be skipped.')
print(f'PyTorch : {torch.__version__}')
n_gpu = torch.cuda.device_count()
print(f'CUDA    : {torch.cuda.is_available()} — {n_gpu} GPU(s) detected')
for g in range(n_gpu):
    vram = torch.cuda.get_device_properties(g).total_memory / 1000000000.0
    print(f'  GPU {g}: {torch.cuda.get_device_name(g)}  ({vram:.1f} GB)')

In [ ]:
@dataclass
class Config:
    kaggle_input_dir: str = '/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw'
    output_dir: str = '/kaggle/working/audiodit_meld'
    audio_cache_dir: str = '/kaggle/working/audio_cache'
    model_id: str = 'meituan-longcat/LongCat-AudioDiT-1B'
    text_encoder_id: str = 'google/flan-t5-small'
    sample_rate: int = 24000
    hop_length: int = 256
    n_fft: int = 1024
    n_mels: int = 80
    max_audio_sec: float = 10.0
    SR = 24000
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'out_proj', 'fc1', 'fc2'])
    batch_size: int = 4
    grad_accum_steps: int = 4
    num_epochs: int = 10
    learning_rate: float = 0.0005
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0
    num_workers: int = 2
    seed: int = 42
    use_fp16: bool = True
    grad_checkpointing: bool = True
    checkpoint_steps: int = 100
    resume_from_checkpoint: str = '/kaggle/input/datasets/aryansharma26/checkpoints/audiodit_meld/audiodit_meld/checkpoints/step_001400'
    num_train_timesteps: int = 1000
    num_inference_steps: int = 50
    guidance_scale: float = 3.5
    emotions: List[str] = field(default_factory=lambda: ['neutral', 'surprise', 'fear', 'sadness', 'joy', 'disgust', 'anger'])
CFG = Config()
torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.audio_cache_dir).mkdir(parents=True, exist_ok=True)
EMOTION2ID = {e: i for i, e in enumerate(CFG.emotions)}
ID2EMOTION = {i: e for e, i in EMOTION2ID.items()}
EMOTION_TOK = {e: f'[{e.upper()}]' for e in CFG.emotions}
print('Config loaded. Emotion tokens:', EMOTION_TOK)
print(f"Checkpoint every {CFG.checkpoint_steps} steps | Resume: '{CFG.resume_from_checkpoint or 'disabled'}'")


## 2 · MELD Dataset — Ingestion & Audio Extraction

In [ ]:
def find_meld_files(root: str) -> Dict[str, Path]:
    root = Path(root)
    mapping = {}
    csv_patterns = {'train': 'train_sent_emo', 'dev': 'dev_sent_emo', 'test': 'test_sent_emo'}
    for split, pat in csv_patterns.items():
        hits = list(root.rglob(f'{pat}.csv'))
        if hits:
            mapping[f'{split}_csv'] = hits[0]
    for mp4 in sorted(root.rglob('*.mp4')):
        parent = mp4.parent
        path_lower = str(parent).lower()
        for split in ['train', 'dev', 'test']:
            key = f'{split}_video_dir'
            if key not in mapping and split in path_lower:
                mapping[key] = parent
    return mapping
MELD_FILES = find_meld_files(CFG.kaggle_input_dir)
print('Found MELD files:')
for k, v in MELD_FILES.items():
    if 'video_dir' in k:
        n_mp4 = len(list(v.glob('*.mp4')))
        print(f'  {k:25s} → {v}  ({n_mp4} .mp4 files)')
    else:
        print(f'  {k:25s} → {v}')


In [ ]:
EMOTION_ALIAS = {'neutral': 'neutral', 'surprise': 'surprise', 'surprised': 'surprise', 'fear': 'fear', 'fearful': 'fear', 'sadness': 'sadness', 'sad': 'sadness', 'joy': 'joy', 'happy': 'joy', 'happiness': 'joy', 'disgust': 'disgust', 'anger': 'anger', 'angry': 'anger'}

def load_meld_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = [c.lower().strip() for c in df.columns]
    rename = {}
    for c in df.columns:
        if 'utterance_id' in c:
            rename[c] = 'utterance_id'
        elif 'emotion' in c:
            rename[c] = 'emotion'
        elif 'dialogue_id' in c:
            rename[c] = 'dialogue_id'
        elif 'utterance' in c:
            rename[c] = 'utterance'
    df = df.rename(columns=rename)
    df['emotion'] = df['emotion'].str.lower().map(EMOTION_ALIAS).fillna('neutral')
    df['emotion_id'] = df['emotion'].map(EMOTION2ID)
    if 'dialogue_id' in df.columns and 'utterance_id' in df.columns:
        df['video_fname'] = df.apply(lambda r: f'dia{int(r.dialogue_id)}_utt{int(r.utterance_id)}.mp4', axis=1)
    return df
dfs = {}
for split in ['train', 'dev', 'test']:
    key = f'{split}_csv'
    if key in MELD_FILES:
        dfs[split] = load_meld_csv(MELD_FILES[key])
        print(f'{split:6s}: {len(dfs[split]):5d} rows — columns: {list(dfs[split].columns)}')
all_df = pd.concat(dfs.values(), ignore_index=True) if dfs else pd.DataFrame()

In [ ]:
AUDIO_DIR = Path('/kaggle/input/datasets/aryansharma26/meld-audio/MELD_audio')

def import_wav_paths(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    split_dir = AUDIO_DIR / split_name
    paths = []
    for _, row in df.iterrows():
        wav = split_dir / row['video_fname'].replace('.mp4', '.wav')
        paths.append(str(wav) if wav.exists() else None)
    df = df.copy()
    df['wav_path'] = paths
    return df
print('Importing pre-extracted WAV paths …')
processed_dfs = {}
for split in ['train', 'dev', 'test']:
    if split in dfs:
        processed_dfs[split] = import_wav_paths(dfs[split], split)
        before = len(processed_dfs[split])
        processed_dfs[split] = processed_dfs[split][processed_dfs[split]['wav_path'].notna()].reset_index(drop=True)
        dropped = before - len(processed_dfs[split])
        print(f"  [{split}] {len(processed_dfs[split])}/{before} rows have WAV files ({('dropped ' + str(dropped) if dropped else 'all matched')})")
    else:
        print(f'Skipping {split} — CSV not found.')
print('\nImport Summary:')
for split, df in processed_dfs.items():
    found = df['wav_path'].notna().sum()
    print(f'{split:5s}: {found}/{len(df)} WAV files found')


## 3 · Exploratory Data Analysis

In [ ]:
def plot_emotion_distribution(dfs_dict: Dict[str, pd.DataFrame], save_path: str) -> None:
    fig, axes = plt.subplots(1, len(dfs_dict), figsize=(6 * len(dfs_dict), 4))
    if len(dfs_dict) == 1:
        axes = [axes]
    palette = plt.cm.Set2(np.linspace(0, 1, len(CFG.emotions)))
    for ax, (split, df) in zip(axes, dfs_dict.items()):
        counts = df['emotion'].value_counts().reindex(CFG.emotions, fill_value=0)
        bars = ax.bar(counts.index, counts.values, color=palette, edgecolor='white', linewidth=0.8)
        ax.set_title(f'MELD — {split.capitalize()} split', fontsize=13, fontweight='bold')
        ax.set_xlabel('Emotion')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=40)
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, str(val), ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')
plot_emotion_distribution(processed_dfs, f'{CFG.output_dir}/emotion_distribution.png')

In [ ]:
def get_audio_durations(df: pd.DataFrame) -> List[float]:
    durations = []
    for wav_path in df['wav_path']:
        try:
            info = sf.info(wav_path)
            durations.append(info.frames / info.samplerate)
        except Exception:
            pass
    return durations

def plot_duration_histograms(dfs_dict: Dict[str, pd.DataFrame], save_path: str) -> None:
    fig, axes = plt.subplots(1, len(dfs_dict), figsize=(6 * len(dfs_dict), 4), sharey=False)
    if len(dfs_dict) == 1:
        axes = [axes]
    for ax, (split, df) in zip(axes, dfs_dict.items()):
        durs = get_audio_durations(df)
        ax.hist(durs, bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
        ax.axvline(np.median(durs), color='#DD8452', linestyle='--', label=f'Median {np.median(durs):.1f}s')
        ax.axvline(CFG.max_audio_sec, color='red', linestyle=':', label=f'Clip limit {CFG.max_audio_sec}s')
        ax.set_title(f'{split.capitalize()} — Duration', fontsize=12, fontweight='bold')
        ax.set_xlabel('Seconds')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')
plot_duration_histograms(processed_dfs, f'{CFG.output_dir}/duration_histograms.png')

## 4 · Dataset & DataLoader

In [ ]:
def wav_to_mel(wav_path: str, sr: int=CFG.sample_rate, n_mels: int=CFG.n_mels, n_fft: int=CFG.n_fft, hop: int=CFG.hop_length, max_sec: float=CFG.max_audio_sec) -> np.ndarray:
    y, file_sr = librosa.load(wav_path, sr=sr, mono=True)
    max_samples = int(max_sec * sr)
    if len(y) > max_samples:
        y = y[:max_samples]
    else:
        y = np.pad(y, (0, max_samples - len(y)))
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels, power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-08)
    return log_mel.astype(np.float32)

def build_emotion_prompt(text: str, emotion: str) -> str:
    token = EMOTION_TOK.get(emotion.lower(), '[NEUTRAL]')
    return f'{token} {text.strip()}'

class MELDDataset(Dataset):

    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        mel = wav_to_mel(row['wav_path'])
        mel_t = torch.from_numpy(mel).unsqueeze(0)
        prompt = build_emotion_prompt(row.get('utterance', ''), row.get('emotion', 'neutral'))
        return {'mel': mel_t, 'prompt': prompt, 'emotion_id': int(row['emotion_id']), 'wav_path': row['wav_path']}

def collate_fn(batch: List[Dict]) -> Dict:
    mels = torch.stack([b['mel'] for b in batch])
    eids = torch.tensor([b['emotion_id'] for b in batch])
    prompts = [b['prompt'] for b in batch]
    wav_paths = [b['wav_path'] for b in batch]
    return {'mel': mels, 'prompt': prompts, 'emotion_id': eids, 'wav_path': wav_paths}
datasets, loaders = ({}, {})
for split, df in processed_dfs.items():
    shuffle = split == 'train'
    datasets[split] = MELDDataset(df)
    loaders[split] = DataLoader(datasets[split], batch_size=CFG.batch_size if split == 'train' else CFG.batch_size * 2, shuffle=shuffle, num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=True, drop_last=split == 'train')
    print(f'{split:6s}: {len(datasets[split]):5d} samples | {len(loaders[split]):4d} batches')
if 'train' in loaders:
    sample = next(iter(loaders['train']))
    print(f"\nSample batch mel shape: {sample['mel'].shape}")
    print(f"Example prompt: {sample['prompt'][0]}")

## 5 · Model Architecture — AudioDiT with LoRA

In [ ]:
class TextEncoder(nn.Module):

    def __init__(self, model_id: str=CFG.text_encoder_id, max_length: int=128):
        super().__init__()
        self.tokenizer = T5Tokenizer.from_pretrained(model_id)
        self.encoder = T5EncoderModel.from_pretrained(model_id)
        self.max_length = max_length
        num_added = self.tokenizer.add_tokens(list(EMOTION_TOK.values()))
        if num_added:
            self.encoder.resize_token_embeddings(len(self.tokenizer))
        for p in self.encoder.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, prompts: List[str], device) -> torch.Tensor:
        enc = self.tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=self.max_length).to(device)
        return self.encoder(**enc).last_hidden_state

def build_scheduler(num_train_timesteps: int=CFG.num_train_timesteps) -> DDPMScheduler:
    return DDPMScheduler(num_train_timesteps=num_train_timesteps, beta_schedule='scaled_linear', clip_sample=False, prediction_type='epsilon')

def find_lora_targets(model: nn.Module, preferred: List[str]) -> List[str]:
    all_linear_basenames = {name.split('.')[-1] for name, mod in model.named_modules() if isinstance(mod, nn.Linear)}
    matched = sorted(all_linear_basenames & set(preferred))
    if matched:
        print(f'  LoRA targets (matched preferred): {matched}')
        return matched
    attn_kw = {'proj', 'query', 'key', 'value', 'attn', 'fc', 'mlp', 'ff', 'gate', 'out'}
    attn_like = sorted((n for n in all_linear_basenames if any((kw in n.lower() for kw in attn_kw))))
    if attn_like:
        print(f'  LoRA targets (attn-like auto-discovered): {attn_like}')
        return attn_like
    targets = sorted(all_linear_basenames)
    print(f'  LoRA targets (all Linear, last-resort): {targets}')
    return targets

class AudioDiTWrapper(nn.Module):

    def __init__(self, model_id: str=CFG.model_id, use_fp16: bool=CFG.use_fp16):
        super().__init__()
        dtype = torch.float16 if use_fp16 else torch.float32
        loaded = False
        try:
            self.unet = AutoModel.from_pretrained(model_id, torch_dtype=dtype, trust_remote_code=True, low_cpu_mem_usage=True)
            self.unet_type = 'auto_remote'
            loaded = True
            print('✅ Loaded via AutoModel (trust_remote_code=True)')
        except Exception as e:
            print(f'AutoModel load attempt: {e}')
        if not loaded:
            try:
                from diffusers import UNet2DConditionModel
                self.unet = UNet2DConditionModel.from_pretrained(model_id, torch_dtype=dtype, low_cpu_mem_usage=True)
                self.unet_type = 'unet2d'
                loaded = True
                print('✅ Loaded via diffusers.UNet2DConditionModel')
            except Exception as e:
                print(f'diffusers load attempt: {e}')
        if not loaded:
            print('⚠  Could not download model — initialising lightweight UNet2DConditionModel as stand-in.')
            from diffusers import UNet2DConditionModel
            self.unet = UNet2DConditionModel(sample_size=64, in_channels=1, out_channels=1, layers_per_block=2, block_out_channels=(64, 128, 256, 256), cross_attention_dim=512, down_block_types=('CrossAttnDownBlock2D',) * 3 + ('DownBlock2D',), up_block_types=('UpBlock2D',) + ('CrossAttnUpBlock2D',) * 3)
            self.unet_type = 'fallback'
        if CFG.grad_checkpointing and hasattr(self.unet, 'enable_gradient_checkpointing'):
            self.unet.enable_gradient_checkpointing()
            print('✅ Gradient checkpointing enabled.')

    def forward(self, noisy_mel, timesteps, encoder_hidden_states):
        out = self.unet(sample=noisy_mel, timestep=timesteps, encoder_hidden_states=encoder_hidden_states)
        if hasattr(out, 'sample'):
            return out.sample
        if isinstance(out, (tuple, list)):
            return out[0]
        return out

def apply_lora(model: AudioDiTWrapper) -> AudioDiTWrapper:
    targets = find_lora_targets(model.unet, CFG.lora_target_modules)
    lora_cfg = LoraConfig(r=CFG.lora_r, lora_alpha=CFG.lora_alpha, lora_dropout=CFG.lora_dropout, target_modules=targets, bias='none')
    model.unet = get_peft_model(model.unet, lora_cfg)
    model.unet.print_trainable_parameters()
    return model
print('Loading models …')
text_encoder = TextEncoder().to(DEVICE)
audiodit = AudioDiTWrapper()
audiodit = apply_lora(audiodit)
audiodit = audiodit.to(DEVICE)
scheduler = build_scheduler()
total_params = sum((p.numel() for p in audiodit.parameters()))
train_params = sum((p.numel() for p in audiodit.parameters() if p.requires_grad))
print(f'\nTotal params : {total_params / 1000000.0:.1f}M')
print(f'Trainable    : {train_params / 1000000.0:.2f}M ({100 * train_params / total_params:.2f}%)')


## 6 · Inference — `generate_emotional_speech`

In [ ]:
def mel_to_wav_griffinlim(log_mel: np.ndarray, sr: int=CFG.sample_rate, n_fft: int=CFG.n_fft, hop: int=CFG.hop_length, n_iter: int=64) -> np.ndarray:
    log_mel = np.clip(log_mel, -1.0, 1.0)
    mel_db = log_mel * 80.0 - 80.0
    mel_pow = librosa.db_to_power(mel_db)
    mel_pow = np.clip(mel_pow, 1e-10, None)
    stft = librosa.feature.inverse.mel_to_stft(mel_pow, sr=sr, n_fft=n_fft, power=2.0)
    wav = librosa.griffinlim(stft, n_iter=n_iter, hop_length=hop)
    wav = np.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)
    peak = np.abs(wav).max()
    if peak > 1e-06:
        wav = wav / peak * 0.95
    return wav.astype(np.float32)

def safe_write_wav(path: str, wav: np.ndarray, sr: int=CFG.sample_rate) -> bool:
    try:
        wav = np.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)
        peak = np.abs(wav).max()
        if peak > 1e-06:
            wav = wav / peak * 0.95
        sf.write(path, wav, sr)
        return True
    except Exception as e:
        print(f'  ⚠  Could not write {path}: {e}')
        return False

def load_weights_for_inference(model, source: str='best') -> str:
    ckpt_base = Path(CFG.output_dir) / 'checkpoints'
    if source == 'final':
        target = Path(CFG.output_dir) / 'lora_weights'
        if not target.exists():
            return '⚠  No final lora_weights dir found.'
        label = 'final lora_weights'
    elif source in ('best', 'latest') and ckpt_base.exists():
        all_ckpts = sorted(ckpt_base.glob('step_*'))
        if not all_ckpts:
            return '⚠  No checkpoints found.'
        if source == 'best':
            best_ckpt, best_mcd = (None, float('inf'))
            for ckpt in all_ckpts:
                ej = ckpt / 'eval_results.json'
                if ej.exists():
                    with open(ej) as f:
                        res = json.load(f)
                    mcds = [r['mcd'] for r in res if not np.isnan(r.get('mcd', float('nan')))]
                    if mcds and (m := (sum(mcds) / len(mcds))) < best_mcd:
                        best_mcd, best_ckpt = (m, ckpt)
            target = best_ckpt or all_ckpts[-1]
            label = f'best ({target.name}, MCD={best_mcd:.2f})' if best_ckpt else f'latest ({target.name})'
        else:
            target = all_ckpts[-1]
            label = f'latest ({target.name})'
    else:
        target = Path(source)
        label = f'explicit ({target.name})'
    unwrapped = accelerator.unwrap_model(model)
    lora_bin = target / 'lora_state_dict.pt'
    if hasattr(unwrapped.unet, 'load_adapter') and (target / 'adapter_config.json').exists():
        unwrapped.unet.load_adapter(str(target), adapter_name='default')
    elif lora_bin.exists():
        sd = torch.load(str(lora_bin), map_location='cpu')
        unwrapped.unet.load_state_dict(sd, strict=False)
    else:
        return f'⚠  No loadable weights in {target}'
    print(f'✅ Inference weights loaded from {label}')
    return label

@torch.no_grad()
def generate_emotional_speech(text: str, emotion_label: str, audiodit_model=None, text_enc=None, sched=None, num_steps: int=CFG.num_inference_steps, guidance_scale: float=CFG.guidance_scale, save_path: Optional[str]=None) -> np.ndarray:
    if audiodit_model is None:
        audiodit_model = audiodit
    if text_enc is None:
        text_enc = text_encoder
    if sched is None:
        sched = scheduler
    audiodit_model.eval()
    device = next(audiodit_model.parameters()).device
    prompt = build_emotion_prompt(text, emotion_label)
    null_prompt = build_emotion_prompt('', 'neutral')
    cond_emb = text_enc([prompt], device)
    uncond_emb = text_enc([null_prompt], device)
    T = int(CFG.max_audio_sec * CFG.sample_rate / CFG.hop_length)
    mel_shape = (1, 1, CFG.n_mels, T)
    latent = torch.randn(mel_shape, device=device, dtype=cond_emb.dtype)
    inference_sched = DDIMScheduler.from_config(sched.config)
    inference_sched.set_timesteps(num_steps)
    for t in inference_sched.timesteps:
        t_batch = t.unsqueeze(0).to(device)
        latent_in = inference_sched.scale_model_input(latent, t)
        noise_cond = audiodit_model(latent_in, t_batch, cond_emb)
        noise_uncond = audiodit_model(latent_in, t_batch, uncond_emb)
        noise_pred = noise_uncond + guidance_scale * (noise_cond - noise_uncond)
        latent = inference_sched.step(noise_pred, t, latent).prev_sample
    generated_mel = latent.squeeze().cpu().float().numpy()
    generated_mel = np.clip(generated_mel, -3.0, 3.0)
    wav = mel_to_wav_griffinlim(generated_mel)
    if save_path:
        safe_write_wav(save_path, wav)
        print(f'Saved → {save_path}')
    return wav


## 7 · Training Loop

### 7-A: Accelerator & Optimizer

In [ ]:
import os
N_GPUS = torch.cuda.device_count()
print(f'Detected {N_GPUS} GPU(s)')
mixed_prec = 'fp16' if CFG.use_fp16 else 'no'
ACCEL_CFG_PATH = '/kaggle/working/accelerate_config.yaml'
accel_cfg_lines = ['compute_environment: LOCAL_MACHINE', 'distributed_type: MULTI_GPU', "downcast_bf16: 'no'", 'gpu_ids: all', 'machine_rank: 0', 'main_training_function: main', f"mixed_precision: '{mixed_prec}'", 'num_machines: 1', f'num_processes: {N_GPUS}', 'rdzv_backend: static', 'same_network: true', 'tpu_env: []', 'tpu_use_cluster: false', 'tpu_use_sudo: false', 'use_cpu: false']
with open(ACCEL_CFG_PATH, 'w') as _f:
    _f.write('\n'.join(accel_cfg_lines) + '\n')
print(f'Accelerate config written → {ACCEL_CFG_PATH}  (num_processes={N_GPUS})')
os.environ['ACCELERATE_CONFIG_FILE'] = ACCEL_CFG_PATH
accelerator = Accelerator(mixed_precision=mixed_prec, gradient_accumulation_steps=CFG.grad_accum_steps, log_with=None)
print(f'Accelerator state : {accelerator.state}')
print(f'Process index     : {accelerator.process_index} / {accelerator.num_processes}')
print(f'Device            : {accelerator.device}')
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, audiodit.parameters()), lr=CFG.learning_rate, betas=(0.9, 0.999), weight_decay=0.001)
train_loader_raw = loaders.get('train')
if train_loader_raw:
    total_steps = len(train_loader_raw) * CFG.num_epochs // CFG.grad_accum_steps
    warmup_steps = int(total_steps * CFG.warmup_ratio)
    lr_scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    audiodit, optimizer, train_loader, lr_scheduler = accelerator.prepare(audiodit, optimizer, train_loader_raw, lr_scheduler)
    text_encoder = text_encoder.to(accelerator.device)
    eff_batch = CFG.batch_size * N_GPUS * CFG.grad_accum_steps
    print(f'\nTotal training steps : {total_steps} | Warmup: {warmup_steps}')
    print(f'Effective batch size : {eff_batch}  ({CFG.batch_size}/GPU × {N_GPUS} GPUs × {CFG.grad_accum_steps} grad_accum)')
else:
    print('No train DataLoader — skipping training setup.')
    train_loader = None


### 7-B: Checkpoint helpers

In [ ]:
CKPT_DIR = Path(CFG.output_dir) / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(audiodit_model, opt, lr_sched, acc: Accelerator, epoch: int, global_step: int) -> Path:
    ckpt_path = CKPT_DIR / f'step_{global_step:06d}'
    ckpt_path.mkdir(parents=True, exist_ok=True)
    unwrapped = acc.unwrap_model(audiodit_model)
    if hasattr(unwrapped.unet, 'save_pretrained'):
        unwrapped.unet.save_pretrained(str(ckpt_path))
    else:
        lora_sd = {k: v for k, v in unwrapped.unet.state_dict().items() if 'lora' in k}
        torch.save(lora_sd, ckpt_path / 'lora_state_dict.pt')
    torch.save(opt.state_dict(), ckpt_path / 'optimizer.pt')
    torch.save(lr_sched.state_dict(), ckpt_path / 'scheduler.pt')
    meta = {'epoch': epoch, 'global_step': global_step, 'checkpoint_steps': CFG.checkpoint_steps}
    with open(ckpt_path / 'meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  💾 Checkpoint saved → {ckpt_path.name}  (ep={epoch}, step={global_step})')
    return ckpt_path

def load_checkpoint(ckpt_path_str: str, audiodit_model, opt, lr_sched, acc: Accelerator):
    ckpt_path = Path(ckpt_path_str)
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')
    print(f'\n🔄 Resuming from checkpoint: {ckpt_path}')
    unwrapped = acc.unwrap_model(audiodit_model)
    lora_bin = ckpt_path / 'lora_state_dict.pt'
    adapter_dir = ckpt_path / 'adapter_model.safetensors'
    if hasattr(unwrapped.unet, 'load_adapter') and (ckpt_path / 'adapter_config.json').exists():
        unwrapped.unet.load_adapter(str(ckpt_path), adapter_name='default')
        print('  ✅ LoRA weights restored via load_adapter()')
    elif lora_bin.exists():
        lora_sd = torch.load(str(lora_bin), map_location='cpu')
        missing, unexpected = unwrapped.unet.load_state_dict(lora_sd, strict=False)
        print(f'  ✅ LoRA weights restored from lora_state_dict.pt (missing={len(missing)}, unexpected={len(unexpected)})')
    else:
        print('  ⚠  No LoRA weights file found in checkpoint — starting with current weights.')
    opt_file = ckpt_path / 'optimizer.pt'
    if opt_file.exists():
        opt.load_state_dict(torch.load(str(opt_file), map_location='cpu'))
        print('  ✅ Optimizer state restored.')
    sched_file = ckpt_path / 'scheduler.pt'
    if sched_file.exists():
        lr_sched.load_state_dict(torch.load(str(sched_file), map_location='cpu'))
        print('  ✅ LR scheduler state restored.')
    meta_file = ckpt_path / 'meta.json'
    if meta_file.exists():
        with open(meta_file) as f:
            meta = json.load(f)
        start_epoch = meta.get('epoch', 1)
        start_global_step = meta.get('global_step', 0)
    else:
        try:
            start_global_step = int(ckpt_path.name.split('_')[-1])
        except ValueError:
            start_global_step = 0
        start_epoch = 1
    print(f'  ▶  Resuming from epoch={start_epoch}, global_step={start_global_step}\n')
    return (start_epoch, start_global_step)

def checkpoint_eval(ckpt_path: Path, text_enc, sched, acc: Accelerator, eval_df=None, n_samples: int=3):
    print(f'  📊 Evaluating checkpoint {ckpt_path.name} …')
    demo_sentences = [("I can't believe this!", 'surprise'), ("I'm so happy today.", 'joy'), ('This is terrible.', 'anger')]
    results = []
    for text, emotion in demo_sentences[:n_samples]:
        try:
            wav = generate_emotional_speech(text=text, emotion_label=emotion, audiodit_model=acc.unwrap_model(audiodit), text_enc=text_enc, sched=sched)
            wav_path = ckpt_path / f'eval_{emotion}.wav'
            sf.write(str(wav_path), wav, CFG.sample_rate)
            mcd = float('nan')
            if eval_df is not None and MCD_AVAILABLE:
                ref_rows = eval_df[eval_df['emotion'] == emotion]
                if len(ref_rows):
                    ref_path = ref_rows.iloc[0]['wav_path']
                    try:
                        mcd_calc = Calculate_MCD(MCD_mode='plain')
                        mcd = mcd_calc.calculate_mcd(ref_path, str(wav_path))
                    except Exception:
                        pass
            results.append({'emotion': emotion, 'mcd': mcd})
            print(f'      [{emotion:8s}] MCD={mcd:.2f}' if not np.isnan(mcd) else f'      [{emotion:8s}] generated')
        except Exception as e:
            print(f'      [{emotion:8s}] eval error: {e}')
    with open(ckpt_path / 'eval_results.json', 'w') as f:
        json.dump(results, f, indent=2)

### 7-C: Diffusion Training Step

In [ ]:
def diffusion_loss_step(audiodit_model, text_enc, noisy_scheduler: DDPMScheduler, batch: Dict, device) -> torch.Tensor:
    mel = batch['mel'].to(device)
    prompts = batch['prompt']
    B = mel.size(0)
    noise = torch.randn_like(mel)
    timesteps = torch.randint(0, noisy_scheduler.config.num_train_timesteps, (B,), device=device, dtype=torch.long)
    noisy_mel = noisy_scheduler.add_noise(mel, noise, timesteps)
    with torch.no_grad():
        text_emb = text_enc(prompts, device)
    pred_noise = audiodit_model(noisy_mel, timesteps, text_emb)
    return F.mse_loss(pred_noise.float(), noise.float())

### 7-D: Main Training Loop

In [ ]:
def train(audiodit_model, text_enc, sched: DDPMScheduler, loader, opt, lr_sched, acc: Accelerator, eval_df=None) -> List[float]:
    step_losses = []
    window = []
    global_step = 0
    start_epoch = 1
    resume = CFG.resume_from_checkpoint.strip() if CFG.resume_from_checkpoint else ''
    if resume:
        start_epoch, global_step = load_checkpoint(resume, audiodit_model, opt, lr_sched, acc)
    steps_per_epoch = len(loader)
    for epoch in range(start_epoch, CFG.num_epochs + 1):
        audiodit_model.train()
        steps_done_this_epoch = global_step % steps_per_epoch if epoch == start_epoch and resume else 0
        for step, batch in enumerate(loader):
            if step < steps_done_this_epoch:
                continue
            with acc.accumulate(audiodit_model):
                loss = diffusion_loss_step(audiodit_model, text_enc, sched, batch, acc.device)
                acc.backward(loss)
                if acc.sync_gradients:
                    acc.clip_grad_norm_(audiodit_model.parameters(), CFG.max_grad_norm)
                opt.step()
                lr_sched.step()
                opt.zero_grad()
            global_step += 1
            loss_val = loss.item()
            step_losses.append(loss_val)
            window.append(loss_val)
            if len(window) > 50:
                window.pop(0)
            if global_step % 10 == 0:
                recent = sum(window) / len(window)
                print(f'Ep {epoch:02d}/{CFG.num_epochs} Step {step + 1:04d}/{steps_per_epoch} [global={global_step}] Loss={recent:.4f} LR={lr_sched.get_last_lr()[0]:.2e}')
            if global_step % CFG.checkpoint_steps == 0:
                ckpt_path = save_checkpoint(audiodit_model, opt, lr_sched, acc, epoch, global_step)
                checkpoint_eval(ckpt_path, text_enc, sched, acc, eval_df=eval_df)
        epoch_losses = step_losses[-steps_per_epoch:]
        epoch_avg = sum(epoch_losses) / max(len(epoch_losses), 1)
        print(f'  ↳ Epoch {epoch} mean loss: {epoch_avg:.4f}')
    return step_losses

### 7-E: Execution

In [ ]:
step_losses = []
_dev = processed_dfs.get('dev')
_test = processed_dfs.get('test')
eval_df_for_ckpt = _dev if _dev is not None else _test
if train_loader is not None:
    print(f"\n{'=' * 60}")
    print(f'Training AudioDiT (LoRA) — {CFG.num_epochs} epochs')
    print(f'Checkpoints every {CFG.checkpoint_steps} steps → {CKPT_DIR}')
    print(f"{'=' * 60}")
    step_losses = train(audiodit, text_encoder, scheduler, train_loader, optimizer, lr_scheduler, accelerator, eval_df=eval_df_for_ckpt)
    print('\n✅ Training complete.')
else:
    print('Skipping training — no data found.')

In [ ]:
CHECKPOINT_TO_EVALUATE = '/kaggle/input/datasets/aryansharma26/checkpoints/checkpoints/step_000900'
label = load_weights_for_inference(audiodit, source=CHECKPOINT_TO_EVALUATE)
print(f'Loaded: {label}')
_ckpt_base = Path(CFG.output_dir) / 'checkpoints'
if not step_losses and _ckpt_base.exists():
    print('Reconstructing loss history from checkpoint metadata …')
    reconstructed = []
    for ckpt in sorted(_ckpt_base.glob('step_*')):
        mj = ckpt / 'meta.json'
        if mj.exists():
            with open(mj) as f:
                m = json.load(f)
            reconstructed.append((m.get('global_step', 0), ckpt))
    if reconstructed:
        print(f'  Found {len(reconstructed)} checkpoints, steps {reconstructed[0][0]}–{reconstructed[-1][0]}')
    else:
        print('  No checkpoint metadata found.')


## 8 · Save LoRA Weights & Config

In [ ]:
def save_lora_weights(audiodit_model, output_dir: str) -> None:
    out = Path(output_dir) / 'lora_weights'
    out.mkdir(parents=True, exist_ok=True)
    unwrapped = accelerator.unwrap_model(audiodit_model)
    if hasattr(unwrapped.unet, 'save_pretrained'):
        unwrapped.unet.save_pretrained(str(out))
        print(f'✅ LoRA weights saved to: {out}')
    else:
        lora_state = {k: v for k, v in unwrapped.unet.state_dict().items() if 'lora' in k}
        torch.save(lora_state, out / 'lora_state_dict.pt')
        print(f"✅ LoRA state_dict saved to: {out / 'lora_state_dict.pt'}")

def save_preprocessing_config(output_dir: str) -> None:
    cfg_dict = {'sample_rate': CFG.sample_rate, 'hop_length': CFG.hop_length, 'n_fft': CFG.n_fft, 'n_mels': CFG.n_mels, 'max_audio_sec': CFG.max_audio_sec, 'emotions': CFG.emotions, 'emotion2id': EMOTION2ID, 'emotion_tokens': EMOTION_TOK, 'text_encoder_id': CFG.text_encoder_id, 'model_id': CFG.model_id, 'lora_r': CFG.lora_r, 'lora_alpha': CFG.lora_alpha, 'checkpoint_evaluated': CHECKPOINT_TO_EVALUATE}
    cfg_path = Path(output_dir) / 'preprocessing_config.json'
    with open(cfg_path, 'w') as f:
        json.dump(cfg_dict, f, indent=2)
    print(f'✅ Config saved to: {cfg_path}')
save_lora_weights(audiodit, CFG.output_dir)
save_preprocessing_config(CFG.output_dir)


## 9 · Visualisation — Loss Curve

In [ ]:
def plot_loss_curve(losses, save_path: str, smooth_window: int=20) -> None:
    if not losses:
        print('No step_losses available.')
        print('If training ran in a previous session, loss values are not persisted.')
        print('Consider logging losses to a CSV during training for cross-session plots.')
        return
    arr = np.array(losses)
    kernel = np.ones(smooth_window) / smooth_window
    smooth = np.convolve(arr, kernel, mode='valid')
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(arr, alpha=0.35, color='#4C72B0', linewidth=0.8, label='Step loss')
    ax.plot(range(smooth_window - 1, len(arr)), smooth, color='#DD8452', linewidth=2.0, label=f'Smoothed ({smooth_window}-step MA)')
    ax.set_title('AudioDiT LoRA Fine-tuning — Loss Curve', fontsize=14, fontweight='bold')
    ax.set_xlabel('Global Step')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')
plot_loss_curve(step_losses, f'{CFG.output_dir}/loss_curve.png')


## 10 · Load best checkpoint for inference

In [ ]:
print(f'Running inference with weights from: {CHECKPOINT_TO_EVALUATE}\n')
test_wav = generate_emotional_speech(text="I can't believe that just happened!", emotion_label='surprise', save_path=f'{CFG.output_dir}/sample_surprise.wav')
print(f'Generated waveform shape: {test_wav.shape} @ {CFG.sample_rate} Hz')


## 11 · Evaluation — MCD & Mel-Spectrogram Comparison

In [ ]:
def compute_mcd_score(ref_wav_path: str, gen_wav: np.ndarray, sr: int=CFG.sample_rate) -> float:
    if not MCD_AVAILABLE:
        return float('nan')
    tmp_gen = '/tmp/_gen_eval.wav'
    safe_write_wav(tmp_gen, gen_wav, sr)
    try:
        return Calculate_MCD(MCD_mode='plain').calculate_mcd(ref_wav_path, tmp_gen)
    except Exception as e:
        print(f'  MCD error: {e}')
        return float('nan')

def run_mcd_evaluation_per_emotion(df: pd.DataFrame) -> pd.DataFrame:
    subset = df.groupby('emotion').sample(n=1, random_state=CFG.seed)
    results = []
    for _, row in subset.iterrows():
        emotion_label = row.get('emotion', 'neutral')
        utterance = row.get('utterance', '')
        gen_wav = generate_emotional_speech(text=utterance, emotion_label=emotion_label)
        mcd = compute_mcd_score(row['wav_path'], gen_wav)
        results.append({'utterance': utterance, 'emotion': emotion_label, 'mcd': mcd})
        print(f'  MCD [{emotion_label:9s}]: {mcd:.2f} dB')
    df_out = pd.DataFrame(results)
    mean_mcd = df_out['mcd'].mean()
    print(f'\nMean MCD across all emotions ({CHECKPOINT_TO_EVALUATE}): {mean_mcd:.2f} dB')
    df_out.to_csv(f'{CFG.output_dir}/mcd_results_per_emotion.csv', index=False)
    return df_out
mcd_df = pd.DataFrame()
if 'test' in processed_dfs and len(processed_dfs['test']) > 0:
    print(f'MCD evaluation per emotion using checkpoint: {CHECKPOINT_TO_EVALUATE}')
    mcd_df = run_mcd_evaluation_per_emotion(processed_dfs['test'])
else:
    print('No test split available for MCD evaluation.')

In [ ]:
def plot_mel_comparison(original_wav_path, text, emotion, save_path) -> None:
    orig_mel = wav_to_mel(original_wav_path)
    gen_wav = generate_emotional_speech(text, emotion)
    tmp_path = '/tmp/_comparison_gen.wav'
    safe_write_wav(tmp_path, gen_wav)
    gen_mel = wav_to_mel(tmp_path)
    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.05)
    for col, (mel, title) in enumerate([(orig_mel, 'Ground Truth'), (gen_mel, f'Generated [{emotion.upper()}]')]):
        ax = fig.add_subplot(gs[col])
        img = ax.imshow(mel, origin='lower', aspect='auto', cmap='magma', interpolation='nearest')
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Time Frames')
        ax.set_ylabel('Mel Bins' if col == 0 else '')
        if col == 1:
            ax.set_yticks([])
        plt.colorbar(img, ax=ax, fraction=0.046, pad=0.04, label='dB')
    suptitle = f'"{text[:60]}…"' if len(text) > 60 else f'"{text}"'
    fig.suptitle(suptitle, fontsize=11, style='italic', y=1.02)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved → {save_path}')
if 'test' in processed_dfs and len(processed_dfs['test']) > 0:
    print(f'Mel comparison using checkpoint: {CHECKPOINT_TO_EVALUATE}')
    for emotion in CFG.emotions:
        sub = processed_dfs['test'][processed_dfs['test']['emotion'] == emotion]
        if len(sub) == 0:
            continue
        row = sub.iloc[0]
        plot_mel_comparison(original_wav_path=row['wav_path'], text=row.get('utterance', 'Sample utterance.'), emotion=emotion, save_path=f'{CFG.output_dir}/mel_comparison_{emotion}.png')
else:
    print('No test split — generating demo spectrograms.')
    demo_text = 'I cannot believe this is happening right now.'
    for emotion in ['joy', 'anger', 'sadness']:
        gen_wav = generate_emotional_speech(demo_text, emotion)
        tmp_path = f'/tmp/demo_{emotion}.wav'
        safe_write_wav(tmp_path, gen_wav)
        mel = wav_to_mel(tmp_path)
        plt.figure(figsize=(7, 3))
        plt.imshow(mel, origin='lower', aspect='auto', cmap='magma')
        plt.title(f'Generated Mel — [{emotion.upper()}]  (ckpt: {CHECKPOINT_TO_EVALUATE})', fontweight='bold')
        plt.xlabel('Time Frames')
        plt.ylabel('Mel Bins')
        plt.colorbar(label='dB')
        plt.tight_layout()
        plt.savefig(f'{CFG.output_dir}/mel_generated_{emotion}.png', dpi=150)
        plt.close()
        print(f'Saved mel for [{emotion}]')


## 12 · Multi-Emotion Inference Demo

In [ ]:
DEMO_TEXT = 'I never thought this would happen to me.'
print(f'Generating: "{DEMO_TEXT}"')
print(f'Checkpoint : {CHECKPOINT_TO_EVALUATE}\n')
for emotion in CFG.emotions:
    out_path = f'{CFG.output_dir}/demo_{emotion}.wav'
    wav = generate_emotional_speech(text=DEMO_TEXT, emotion_label=emotion, save_path=out_path)
    duration = len(wav) / CFG.sample_rate
    print(f'  [{emotion:10s}] → {out_path}  ({duration:.2f}s)')
print('\n✅ Inference demo complete.')


## 12 · Summary & Output Manifest

In [ ]:
import os
output_files = []
for root, dirs, files in os.walk(CFG.output_dir):
    for fname in sorted(files):
        fpath = Path(root) / fname
        size_kb = fpath.stat().st_size / 1024
        output_files.append({'file': str(fpath.relative_to(CFG.output_dir)), 'size_kb': f'{size_kb:.1f}'})
manifest_df = pd.DataFrame(output_files)
print(f'Output directory: {CFG.output_dir}\n')
print(manifest_df.to_string(index=False))
manifest_df.to_csv(f'{CFG.output_dir}/output_manifest.csv', index=False)
print('\n✅ Notebook complete. All outputs written to /kaggle/working/audiodit_meld/')